# AI Risk Manager — EDA
## Phase 1: Return Risk Scorer

This notebook explores `data/raw/returns.csv` before training the model.  
Goal: understand the data, confirm signals are present, spot any issues.

**Questions we answer:**
1. Is the class balance correct? (85% legit / 15% abusive)
2. Which features differ most between abusive and legitimate returns?
3. How are key numeric features distributed?
4. What does the correlation heatmap show?
5. Are there any missing values or outliers to handle?

---
## 1. Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

DATA_PATH = '../data/raw/returns.csv'
df = pd.read_csv(DATA_PATH)

print(f'Shape : {df.shape}')
print(f'Columns : {df.columns.tolist()}')
df.head()

---
## 2. Basic Info

In [ ]:
print('=== Data Types ===')
print(df.dtypes)
print('\n=== Missing Values ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else 'None ✓')
print('\n=== Numeric Summary ===')
df.describe().round(2)

---
## 3. Class Balance
Expected: ~85% legitimate (0), ~15% abusive (1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
counts = df['is_label_abusive'].value_counts()
labels = ['Legitimate', 'Abusive']
colors = ['#4CAF50', '#F44336']
bars = axes[0].bar(labels, counts.values, color=colors, width=0.5, edgecolor='white')
for bar, count in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f'{count:,}\n({count/len(df)*100:.1f}%)',
                 ha='center', va='bottom', fontweight='bold')
axes[0].set_title('Class Balance', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Record Count')
axes[0].set_ylim(0, max(counts.values) * 1.15)

# Pie chart
axes[1].pie(counts.values, labels=labels, colors=colors,
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Class Distribution', fontsize=13, fontweight='bold')

plt.suptitle('Target Variable: is_label_abusive', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 4. Abuse Pattern Breakdown
Which of the 4 abuse patterns is most common?

In [ ]:
pattern_counts = df['abuse_pattern'].value_counts()

fig, ax = plt.subplots(figsize=(10, 4))
palette = ['#9E9E9E', '#EF5350', '#FF7043', '#FFA726', '#AB47BC']
bars = ax.barh(pattern_counts.index, pattern_counts.values,
               color=palette[:len(pattern_counts)], edgecolor='white')
for bar, count in zip(bars, pattern_counts.values):
    ax.text(bar.get_width() + 30, bar.get_y() + bar.get_height()/2,
            f'{count:,}  ({count/len(df)*100:.1f}%)',
            va='center', fontsize=10)
ax.set_title('Abuse Pattern Distribution (including "none" = legitimate)', 
             fontsize=13, fontweight='bold')
ax.set_xlabel('Record Count')
ax.set_xlim(0, max(pattern_counts.values) * 1.15)
plt.tight_layout()
plt.show()

---
## 5. Key Feature Comparison: Abusive vs Legitimate
This is the most important EDA section — are our signals actually visible?

In [ ]:
legit  = df[df['is_label_abusive'] == False]
abusive = df[df['is_label_abusive'] == True]

print('=== Mean feature values: Abusive vs Legitimate ===')
numeric_features = [
    'return_rate_lifetime', 'return_rate_30d', 'days_to_return',
    'order_value', 'order_value_normalized', 'customer_account_age_days',
    'category_risk_score', 'return_reason_risk', 'merchant_return_rate'
]
comparison = pd.DataFrame({
    'Legitimate (mean)': legit[numeric_features].mean(),
    'Abusive (mean)':    abusive[numeric_features].mean(),
}).round(4)
comparison['Ratio (Abusive/Legit)'] = (
    comparison['Abusive (mean)'] / comparison['Legitimate (mean)']
).round(2)
comparison.style.background_gradient(subset=['Ratio (Abusive/Legit)'], cmap='RdYlGn_r')

In [ ]:
# Boxplots for top numeric features
features_to_plot = [
    'return_rate_lifetime', 'days_to_return',
    'order_value', 'customer_account_age_days'
]

# seaborn needs string keys when column is boolean
plot_df = df.copy()
plot_df['label'] = plot_df['is_label_abusive'].map({False: 'Legitimate', True: 'Abusive'})

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, feat in zip(axes, features_to_plot):
    sns.boxplot(
        data=plot_df, x='label', y=feat,
        palette={'Legitimate': '#4CAF50', 'Abusive': '#F44336'},
        order=['Legitimate', 'Abusive'],
        ax=ax, width=0.5
    )
    ax.set_title(feat, fontweight='bold', fontsize=11)
    ax.set_xlabel('')

plt.suptitle('Feature Distribution: Legitimate vs Abusive', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 6. Binary Feature Rates
For boolean features — what % of each group has the flag set?

In [ ]:
binary_features = [
    'support_contacted', 'images_submitted',
    'is_near_deadline', 'is_same_day_return',
    'is_new_account', 'is_first_order',
    'no_images_high_value', 'no_support_contact'
]

rate_data = pd.DataFrame({
    'Legitimate': legit[binary_features].mean() * 100,
    'Abusive':    abusive[binary_features].mean() * 100
}).round(1)

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(binary_features))
width = 0.35

bars1 = ax.bar(x - width/2, rate_data['Legitimate'], width,
               label='Legitimate', color='#4CAF50', alpha=0.85, edgecolor='white')
bars2 = ax.bar(x + width/2, rate_data['Abusive'], width,
               label='Abusive', color='#F44336', alpha=0.85, edgecolor='white')

ax.set_title('Binary Feature Rates: Legitimate vs Abusive (%)',
             fontsize=13, fontweight='bold')
ax.set_ylabel('% of group with flag = True')
ax.set_xticks(x)
ax.set_xticklabels(binary_features, rotation=30, ha='right')
ax.legend()
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
plt.tight_layout()
plt.show()

print(rate_data.to_string())

---
## 7. Category & Reason Distribution by Label

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Product category
cat_data = df.groupby(['product_category', 'is_label_abusive']).size().unstack(fill_value=0)
cat_pct  = cat_data.div(cat_data.sum(axis=1), axis=0) * 100
cat_pct.columns = ['Legitimate', 'Abusive']
cat_pct.sort_values('Abusive', ascending=True).plot(
    kind='barh', ax=axes[0],
    color=['#4CAF50', '#F44336'], edgecolor='white', alpha=0.85
)
axes[0].set_title('Abuse Rate by Product Category (%)', fontweight='bold')
axes[0].set_xlabel('%')
axes[0].xaxis.set_major_formatter(mticker.PercentFormatter())
axes[0].legend(loc='lower right')

# Return reason
reason_data = df.groupby(['return_reason', 'is_label_abusive']).size().unstack(fill_value=0)
reason_pct  = reason_data.div(reason_data.sum(axis=1), axis=0) * 100
reason_pct.columns = ['Legitimate', 'Abusive']
reason_pct.sort_values('Abusive', ascending=True).plot(
    kind='barh', ax=axes[1],
    color=['#4CAF50', '#F44336'], edgecolor='white', alpha=0.85
)
axes[1].set_title('Abuse Rate by Return Reason (%)', fontweight='bold')
axes[1].set_xlabel('%')
axes[1].xaxis.set_major_formatter(mticker.PercentFormatter())
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()

---
## 8. Numeric Feature Distributions

In [ ]:
dist_features = [
    'return_rate_lifetime', 'return_rate_30d',
    'days_to_return', 'order_value',
    'customer_account_age_days', 'order_value_normalized'
]

if 'label' not in df.columns:
    df['label'] = df['is_label_abusive'].map({False: 'Legitimate', True: 'Abusive'})

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for ax, feat in zip(axes, dist_features):
    sns.histplot(
        data=df, x=feat, hue='label',
        palette={'Legitimate': '#4CAF50', 'Abusive': '#F44336'},
        hue_order=['Legitimate', 'Abusive'],
        kde=True, alpha=0.5, bins=40, ax=ax
    )
    ax.set_title(feat, fontweight='bold', fontsize=11)
    ax.set_xlabel('')

plt.suptitle('Feature Distributions: Legitimate vs Abusive',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 9. Correlation Heatmap
Which features are most correlated with `is_label_abusive`?

In [ ]:
model_features = [
    'return_rate_lifetime', 'return_rate_30d', 'days_to_return',
    'is_near_deadline', 'is_same_day_return', 'category_risk_score',
    'is_first_order', 'is_new_account', 'no_images_high_value',
    'no_support_contact', 'return_reason_risk', 'order_value_normalized',
    'merchant_return_rate', 'order_value', 'customer_account_age_days',
    'is_label_abusive'
]

corr = df[model_features].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, vmin=-1, vmax=1,
    linewidths=0.5, ax=ax,
    annot_kws={'size': 8}
)
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Top correlations with target
print('\n=== Correlation with is_label_abusive (sorted) ===')
target_corr = corr['is_label_abusive'].drop('is_label_abusive').sort_values(key=abs, ascending=False)
print(target_corr.round(4).to_string())

---
## 10. Order Value Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Order value by category
cat_order = df.groupby('product_category')['order_value'].median().sort_values(ascending=False)
axes[0].bar(cat_order.index, cat_order.values,
            color='#42A5F5', edgecolor='white', alpha=0.85)
axes[0].set_title('Median Order Value by Category (₹)', fontweight='bold')
axes[0].set_ylabel('Median Order Value (INR)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x:,.0f}'))

# Order value by abuse label
df.boxplot(column='order_value', by='is_label_abusive',
           ax=axes[1], patch_artist=True)
axes[1].set_title('Order Value: Legitimate vs Abusive', fontweight='bold')
axes[1].set_xlabel('')
axes[1].set_xticklabels(['Legitimate', 'Abusive'])
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x:,.0f}'))
plt.suptitle('')

plt.tight_layout()
plt.show()

print('\nOrder value percentiles by label:')
print(df.groupby('is_label_abusive')['order_value'].describe().round(0).to_string())

---
## 11. EDA Summary

Key findings from the data exploration:

In [ ]:
print('=' * 60)
print('  EDA SUMMARY — Key Findings')
print('=' * 60)

# Class balance
abuse_pct = df['is_label_abusive'].mean() * 100
print(f'\n1. CLASS BALANCE')
print(f'   Abusive rate : {abuse_pct:.1f}%  (target: 15%)')

# Strongest signals
print(f'\n2. STRONGEST SIGNALS (correlation with target)')
for feat, corr_val in target_corr.head(6).items():
    print(f'   {feat:<30} {corr_val:+.4f}')

# Return rate gap
lr = legit['return_rate_lifetime'].mean()
ar = abusive['return_rate_lifetime'].mean()
print(f'\n3. RETURN RATE GAP')
print(f'   Legitimate avg return rate  : {lr:.3f}')
print(f'   Abusive avg return rate     : {ar:.3f}')
print(f'   Ratio (abusive/legit)       : {ar/lr:.1f}x')

# Image submission gap
li = legit['images_submitted'].mean() * 100
ai = abusive['images_submitted'].mean() * 100
print(f'\n4. IMAGE SUBMISSION')
print(f'   Legitimate submit images    : {li:.1f}%')
print(f'   Abusive submit images       : {ai:.1f}%')

# Support contact gap
ls = legit['support_contacted'].mean() * 100
as_ = abusive['support_contacted'].mean() * 100
print(f'\n5. SUPPORT CONTACT')
print(f'   Legitimate contact support  : {ls:.1f}%')
print(f'   Abusive contact support     : {as_:.1f}%')

print(f'\n6. MISSING VALUES    : None ✓')
print(f'   TOTAL RECORDS     : {len(df):,}')
print(f'   TOTAL FEATURES    : {len(df.columns)}')
print(f'\n  → Data is clean. Signals are strong. Ready for feature engineering.')
print('=' * 60)